***Regression model***
**For the regression model, I will predict the fare amount**
I will use the following features:
- passenger count
- trip distance
- pick up hour
- pick up day of the week

I will use the `drop_tip_applied` method from data_preprocessing to remove the tip_applied column.

For the ML model, I will use the Random Forest regressor.

**Metrics**
- MSE (Mean Squared Error) 

In [2]:
from data_preprocessing import load_and_process, drop_tip_applied, train_val_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error
import pandas as pd

In [3]:
data = load_and_process('dataset/green_tripdata_2021-01.parquet')
data = drop_tip_applied(data)
data.head()

,passenger_count,trip_distance,fare_amount,pickup_hour,pickup_day_of_week
0,1.0,1.01,5.5,0,4
1,1.0,2.53,10.0,0,4
2,1.0,1.12,6.0,0,4
3,1.0,1.99,8.0,23,3
7,6.0,0.45,3.5,0,4


In [18]:
train, val, test = train_val_test_split(data, random_state=120)

In [19]:
print(f"train size: {train.shape}, val size: {val.shape}, test size: {test.shape}")

train size: (26473, 5), val size: (5673, 5), test size: (5673, 5)


In [20]:
# Feature columns (per specification) and target
FEATURE_COLUMNS = [
    "passenger_count",
    "trip_distance",
    "pickup_hour",
    "pickup_day_of_week",
]
TARGET_COLUMN = "fare_amount"

X_train = train[FEATURE_COLUMNS]
X_val = val[FEATURE_COLUMNS]
X_test = test[FEATURE_COLUMNS]

y_train = train[TARGET_COLUMN]
y_val = val[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

In [21]:
X_train.head()

,passenger_count,trip_distance,pickup_hour,pickup_day_of_week
12907,1.0,3.95,13,0
19410,1.0,0.95,17,4
9067,1.0,0.13,12,4
19211,1.0,2.10,15,4
25791,2.0,0.81,0,3


In [22]:
param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 6, 7, 8, 10],
    "random_state": [100],
}

In [23]:
results = []
for params in ParameterGrid(param_grid):
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    results.append({
        **params,
        "train_mse": mean_squared_error(y_train, y_train_pred),
        "val_mse": mean_squared_error(y_val, y_val_pred),
    })

# Store results and best model (by val MSE - lower is better)
best_idx = min(range(len(results)), key=lambda i: results[i]["val_mse"])
best_params = {k: v for k, v in results[best_idx].items() if k in param_grid}
best_mse = results[best_idx]["val_mse"]
best_model = RandomForestRegressor(**best_params).fit(X_train, y_train)

In [24]:
results_df = pd.DataFrame(results).sort_values("val_mse")
results_df[["n_estimators", "max_depth", "val_mse", "train_mse"]]

,n_estimators,max_depth,val_mse,train_mse
7,300,6,21.222474,16.198193
6,200,6,21.270781,16.260184
5,100,6,21.375390,16.314964
15,300,8,21.562108,11.823793
14,200,8,21.565767,11.842785
4,50,6,21.583641,16.188920
13,100,8,21.727828,11.869184
11,300,7,21.758140,13.193063
10,200,7,21.789847,13.200801
19,300,10,21.801346,9.346310


In [25]:
print("Best parameters:", best_params)
print("Best validation MSE:", best_mse)
y_val_pred = best_model.predict(X_val)
print("Validation MSE:", mean_squared_error(y_val, y_val_pred))

Best parameters: {'max_depth': 6, 'n_estimators': 300, 'random_state': 100}
Best validation MSE: 21.222474305347177
Validation MSE: 21.222474305347177


In [26]:
# Evaluate best model on test set
y_test_pred = best_model.predict(X_test)

print("Test set metrics:")
print("MSE:", mean_squared_error(y_test, y_test_pred))

Test set metrics:
MSE: 21.07269158044932
